## RQ3: build the modeling dataset

This notebook starts by assembling a row-level dataset for the prediction task: cleaned flight records plus the fixed historical route-cluster features and airport-network features.


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
import sys

PROJECT_DIR = Path.cwd().resolve()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.config import CLEAN_PARQUET, PROJECT_ROOT
from src.data_loading import load_flights

results_dir = PROJECT_ROOT / "results" / "tables"

# Fixed historical features from the Jan-Jun training window for RQ1 and RQ2.
route_lookup = pd.read_csv(results_dir / "rq1_route_clusters_jan_jun.csv")
route_rates = pd.read_csv(results_dir / "rq1_cluster_delay_rates_jan_jun.csv")
airport_features = pd.read_csv(results_dir / "03_airports.csv")

# Keep only completed flights for the prediction target: an arrival late flag is meaningful only for flights that actually arrived.
flights = load_flights(
    CLEAN_PARQUET,
    columns=[
        "month",
        "fl_date",
        "day_of_week",
        "day_of_month",
        "op_unique_carrier",
        "origin",
        "dest",
        "origin_state_nm",
        "dest_state_nm",
        "crs_dep_time",
        "crs_arr_time",
        "distance",
        "crs_elapsed_time",
        "status",
        "cancelled",
        "diverted",
        "arr_delay",
        "dep_delay",
    ],
)

flights = flights[flights["status"] == "completed"].copy().reset_index(drop=True)

# Outcome definitions used later in the prediction task.
flights["target_late_arrival_15"] = (flights["arr_delay"] > 15).astype(int)
flights["target_late_departure_15"] = (flights["dep_delay"] > 15).astype(int)

# Route-level historical features from Jan-Jun clustering (fixed across folds).
flights = flights.merge(route_lookup, on=["origin", "dest", "op_unique_carrier"], how="left")
flights = flights.merge(route_rates, on="route_cluster", how="left")

# Origin and destination airport network features from the airport graph analysis.
origin_features = airport_features.rename(columns={
    "airport": "origin",
    "louvain": "origin_community",
    "betweenness": "origin_betweenness",
    "betweenness_hops": "origin_betweenness_hops",
    "degree": "origin_degree",
    "flights": "origin_flights",
    "late_arrival_share": "origin_late_arrival_share",
    "departing_flights": "origin_departing_flights",
    "late_departure_share": "origin_late_departure_share",
})

dest_features = airport_features.rename(columns={
    "airport": "dest",
    "louvain": "dest_community",
    "betweenness": "dest_betweenness",
    "betweenness_hops": "dest_betweenness_hops",
    "degree": "dest_degree",
    "flights": "dest_flights",
    "late_arrival_share": "dest_late_arrival_share",
    "departing_flights": "dest_departing_flights",
    "late_departure_share": "dest_late_departure_share",
})

flights = flights.merge(origin_features[[
    "origin", "origin_community", "origin_betweenness", "origin_betweenness_hops",
    "origin_degree", "origin_flights", "origin_late_arrival_share",
    "origin_departing_flights", "origin_late_departure_share"
]], on="origin", how="left")

flights = flights.merge(dest_features[[
    "dest", "dest_community", "dest_betweenness", "dest_betweenness_hops",
    "dest_degree", "dest_flights", "dest_late_arrival_share",
    "dest_departing_flights", "dest_late_departure_share"
]], on="dest", how="left")

# Keep the dataset in a clean row-wise format for later temporal train/test splits.
flights["route_cluster"] = flights["route_cluster"].fillna("unknown")
flights["historical_delay_rate"] = flights["historical_delay_rate"].astype(float)
flights["n_historical_flights"] = flights["n_historical_flights"].fillna(0).astype(int)

# A few useful convenience columns for the EBM model.
flights["is_cancelled"] = (flights["cancelled"] == 1).astype(int)
flights["is_diverted"] = (flights["diverted"] == 1).astype(int)

print(f"RQ3 modeling dataframe rows: {len(flights):,}")
print(f"Features available: {len(flights.columns)} columns")
print(f"Target rate: {flights['target_late_arrival_15'].mean():.3%}")
print(f"Columns: {list(flights.columns)}")

rq3_df = flights.copy()


C:\Users\huste\AppData\Local\Temp\ipykernel_38208\3847929173.py:95: Pandas4Warning: 'str' is not supported as a fill value for float64 dtype. In a future version, calling fillna with an incompatible value will raise. Explicitly cast to a common dtype before filling.
  flights["route_cluster"] = flights["route_cluster"].fillna("unknown")


RQ3 modeling dataframe rows: 6,965,246
Features available: 41 columns
Target rate: 20.148%
Columns: ['month', 'fl_date', 'day_of_week', 'day_of_month', 'op_unique_carrier', 'origin', 'dest', 'origin_state_nm', 'dest_state_nm', 'crs_dep_time', 'crs_arr_time', 'distance', 'crs_elapsed_time', 'status', 'cancelled', 'diverted', 'arr_delay', 'dep_delay', 'target_late_arrival_15', 'target_late_departure_15', 'route_cluster', 'historical_delay_rate', 'n_historical_flights', 'origin_community', 'origin_betweenness', 'origin_betweenness_hops', 'origin_degree', 'origin_flights', 'origin_late_arrival_share', 'origin_departing_flights', 'origin_late_departure_share', 'dest_community', 'dest_betweenness', 'dest_betweenness_hops', 'dest_degree', 'dest_flights', 'dest_late_arrival_share', 'dest_departing_flights', 'dest_late_departure_share', 'is_cancelled', 'is_diverted']
